# INOVAAPPS 2026 · Exploração da carteira

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from plotly.subplots import make_subplots
from IPython.display import display

from dados import (CORES, CORES_NPS, METRICAS, ORDEM_NPS, ORDEM_STATUS, ROTULOS, antecedencia,
                   auc_separacao, carregar, fmt_brl, fmt_mes, media_janela_final, perfil_relativo, suavizar)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["inova"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, system-ui, sans-serif", size=13, color="#2b2b29"),
    title=dict(font=dict(size=17), x=0.01, xanchor="left"),
    separators=",.", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="#ffffff", bordercolor="#c9c7c0", font=dict(size=12, color="#2b2b29")),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="right", x=1, title_text=""),
    margin=dict(l=70, r=30, t=110, b=60),
    xaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
    yaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
))
pio.templates.default = "plotly_white+inova"
CINZA, CINZA_CLARO = "#8a8984", "#eeede9"

def rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[:2], 16)},{int(h[2:4], 16)},{int(h[4:], 16)},{a})"

def br(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",")

MES0 = pd.Period("2025-01", "M")
MESES = pd.period_range("2025-01", "2026-06", freq="M")
def idx(p):                       # Period → posição no eixo (jan/2025 = 0)
    return (p - MES0).n

def eixo_meses(fig, passo=1, **kw):
    fig.update_xaxes(tickvals=list(range(0, len(MESES), passo)),
                     ticktext=[fmt_mes(p) for p in MESES[::passo]],
                     range=[-0.6, len(MESES) - 0.4], **kw)

def eixo_relativo(fig, ini=-12, **kw):
    fig.update_xaxes(tickvals=list(range(ini, 0)), ticktext=[f"{m}".replace("-", "−") for m in range(ini, 0)], **kw)

d = carregar()
cli, at, nps = d.clientes, d.atendimento, d.nps
assert (len(cli), (cli.status == "Cancelou").sum(), len(at), len(nps)) == (80, 22, 1295, 422)

## 1. Carteira

In [2]:
canc = cli[cli.status == "Cancelou"]
kpis = [("Clientes", len(cli), "", "", ",.0f"),
        ("Cancelaram", len(canc), "", "", ",.0f"),
        ("Churn no período", 100 * len(canc) / len(cli), "", "%", ".1f"),
        (f"Receita mensal perdida<br><span style='font-size:12px;color:{CINZA}'>"
         f"{br(100 * canc.valor_mensal.sum() / cli.valor_mensal.sum(), 1)}% da receita da carteira</span>",
         canc.valor_mensal.sum(), "R$ ", "", ",.0f")]
fig = go.Figure([go.Indicator(mode="number", value=v, title=dict(text=t, font_size=14),
                              number=dict(prefix=pre, suffix=suf, valueformat=f, font_size=44),
                              domain=dict(row=0, column=i)) for i, (t, v, pre, suf, f) in enumerate(kpis)])
fig.update_layout(grid=dict(rows=1, columns=4), height=230, margin=dict(t=70, b=10),
                  title="Carteira de jan/2025 a jun/2026")
fig.show()

In [3]:
geral = 100 * (cli.status == "Cancelou").mean()
fig = make_subplots(rows=1, cols=3, subplot_titles=["Segmento", "Porte", "Plano"], horizontal_spacing=0.12)
for c, col in enumerate(["segmento", "porte", "plano"], start=1):
    t = (cli.groupby(col, observed=True).agg(n=("cliente_id", "size"), canc=("status", lambda s: (s == "Cancelou").sum()))
            .assign(pct=lambda x: 100 * x.canc / x.n))
    t = t.sort_values("pct") if col == "segmento" else t.iloc[::-1]
    fig.add_trace(go.Bar(y=t.index.astype(str), x=t.pct, orientation="h", marker_color=CORES["Cancelou"],
                         customdata=t[["canc", "n"]], texttemplate="%{x:.0f}% (%{customdata[0]}/%{customdata[1]})",
                         textposition="outside", cliponaxis=False, showlegend=False,
                         hovertemplate="<b>%{y}</b><br>Churn: %{x:.1f}%<br>%{customdata[0]} de %{customdata[1]} clientes<extra></extra>"),
                  row=1, col=c)
    fig.add_vline(x=geral, line=dict(color=CINZA, dash="dot", width=1.5), row=1, col=c)
fig.add_annotation(x=geral, y=1, xref="x", yref="paper", yanchor="bottom", showarrow=False,
                   text=f"carteira: {br(geral, 1)}%", font=dict(size=11, color=CINZA))
fig.update_xaxes(title_text="% de clientes que cancelaram", range=[0, 75], ticksuffix="%")
fig.update_layout(height=380, title="Churn por segmento, porte e plano (linha pontilhada = média da carteira)")
fig.show()

In [4]:
fig = go.Figure()
for st in ORDEM_STATUS[::-1]:
    s = cli[cli.status == st]
    fig.add_trace(go.Box(x=s.valor_mensal, name=f"{st} ({len(s)})", marker_color=CORES[st], boxpoints="all",
                         jitter=0.5, pointpos=0, marker=dict(size=7, opacity=0.7, line=dict(color="white", width=1)),
                         line_width=1.5, fillcolor=rgba(CORES[st], 0.12), customdata=s[["cliente_id", "plano"]],
                         hovertemplate="<b>%{customdata[0]}</b> · %{customdata[1]}<br>R$ %{x:,.0f}/mês<extra></extra>"))
fig.update_layout(height=320, showlegend=False, title="Valor mensal do contrato: quem cancelou × quem continua ativo")
fig.update_xaxes(title_text="Valor mensal (R$)", tickprefix="R$ ", tickformat=",.0f")
fig.show()

## 2. Cancelamentos no tempo

In [5]:
por_mes = (canc.groupby("mes_cancelamento").agg(n=("cliente_id", "size"), receita=("valor_mensal", "sum"),
                                                ids=("cliente_id", ", ".join))
               .reindex(MESES).fillna({"n": 0, "receita": 0, "ids": "—"}))
por_mes["acum"] = por_mes.receita.cumsum()
x, rot = [idx(p) for p in por_mes.index], [fmt_mes(p) for p in por_mes.index]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=["Clientes que cancelaram no mês", "Receita mensal perdida, acumulada"])
fig.add_trace(go.Bar(x=x, y=por_mes.n, marker_color=CORES["Cancelou"], customdata=np.c_[rot, por_mes.ids, por_mes.receita],
                     text=por_mes.n.where(por_mes.n > 0).map(lambda v: "" if pd.isna(v) else f"{v:.0f}"), textposition="outside",
                     cliponaxis=False, hovertemplate="<b>%{customdata[0]}</b><br>%{y:.0f} cancelamento(s)<br>%{customdata[1]}"
                     "<br>R$ %{customdata[2]:,.0f}/mês<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=por_mes.acum, mode="lines+markers", line=dict(color=CORES["Cancelou"], width=2, shape="hv"),
                         fill="tozeroy", fillcolor=rgba(CORES["Cancelou"], 0.10), marker_size=6, customdata=rot,
                         hovertemplate="<b>%{customdata}</b><br>Acumulado: R$ %{y:,.0f}/mês<extra></extra>"), row=2, col=1)
fig.add_annotation(x=x[-1], y=por_mes.acum.iloc[-1], text=f"<b>{fmt_brl(por_mes.acum.iloc[-1])}/mês</b>", showarrow=False,
                   xanchor="right", yanchor="bottom", yshift=6, row=2, col=1)
eixo_meses(fig, tickangle=-45)
fig.update_yaxes(title_text="Cancelamentos", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="R$ por mês", tickprefix="R$ ", tickformat=",.0f", row=2, col=1)
fig.update_layout(height=560, showlegend=False,
                  title=f"{len(canc)} cancelamentos entre {fmt_mes(canc.mes_cancelamento.min())} e "
                        f"{fmt_mes(canc.mes_cancelamento.max())} — receita recorrente perdida por mês de saída")
fig.show()

## 3. Sinais antes da saída

In [6]:
sm = suavizar(at)                                  # média móvel de 3 meses por cliente (só passado)
fig = make_subplots(rows=3, cols=3, horizontal_spacing=0.07, vertical_spacing=0.12, shared_xaxes=True,
                    subplot_titles=[" "] * len(METRICAS))
lead = {}
for i, m in enumerate(METRICAS):
    r, c = divmod(i, 3)
    p = perfil_relativo(sm, m, -12, -1)
    lead[m] = antecedencia(p)
    for st in ORDEM_STATUS:
        q = p[p.status == st]
        fig.add_trace(go.Scatter(x=list(q.mes_relativo) + list(q.mes_relativo[::-1]), y=list(q.p75) + list(q.p25[::-1]),
                                 fill="toself", fillcolor=rgba(CORES[st], 0.15), line_width=0, hoverinfo="skip",
                                 legendgroup=st, showlegend=False), row=r + 1, col=c + 1)
        fig.add_trace(go.Scatter(x=q.mes_relativo, y=q.mediana, mode="lines+markers", name=f"{st} (mediana e faixa P25–P75)",
                                 line=dict(color=CORES[st], width=2), marker_size=5, legendgroup=st, showlegend=i == 0,
                                 customdata=q[["p25", "p75", "n"]],
                                 hovertemplate=f"<b>{st}</b> · mês %{{x}}<br>Mediana: %{{y:,.1f}}<br>"
                                               "P25–P75: %{customdata[0]:,.1f} – %{customdata[1]:,.1f}<br>n = %{customdata[2]} clientes<extra></extra>"),
                      row=r + 1, col=c + 1)
    sub = f"separa desde o mês −{lead[m]}" if lead[m] else "não separa"
    fig.layout.annotations[i].text = f"<b>{ROTULOS[m]}</b><br><span style='font-size:11px;color:{CINZA}'>{sub}</span>"
fig.add_vrect(x0=-3.5, x1=-0.5, fillcolor=CINZA_CLARO, opacity=0.6, line_width=0, layer="below", row="all", col="all")
eixo_relativo(fig, tickfont_size=10)
fig.update_xaxes(title_text="Meses até a saída (−1 = último mês com dados)", row=3)
fig.update_layout(height=920, margin=dict(t=150), legend=dict(y=1.06),
                  title="Como cada métrica evolui nos 12 meses antes da saída — média móvel de 3 meses por cliente<br>"
                        f"<span style='font-size:12px;color:{CINZA}'>Ativos alinhados a jul/2026 · faixa cinza = 3 meses finais · "
                        "“separa desde” = mediana de quem cancelou fora da faixa P25–P75 dos ativos até o fim</span>")
fig.show()

## 4. Quão bem separa

In [7]:
fim = media_janela_final(at, 3)
aucs = {m: auc_separacao(fim, m) for m in METRICAS}
ordem = sorted(METRICAS, key=lambda m: -abs(aucs[m] - 0.5))
fig = make_subplots(rows=3, cols=3, horizontal_spacing=0.08, vertical_spacing=0.13, subplot_titles=[" "] * len(ordem))
for i, m in enumerate(ordem):
    r, c = divmod(i, 3)
    for st in ORDEM_STATUS:
        s = fim[fim.status == st]
        fig.add_trace(go.Box(x=[st] * len(s), y=s[m], name=st, marker_color=CORES[st], boxpoints="all", jitter=0.45,
                             pointpos=0, marker=dict(size=5, opacity=0.6), line_width=1.5, fillcolor=rgba(CORES[st], 0.12),
                             legendgroup=st, showlegend=i == 0, customdata=s[["cliente_id"]],
                             hovertemplate=f"<b>%{{customdata[0]}}</b> · {st}<br>{ROTULOS[m]}: %{{y:,.1f}}<extra></extra>"),
                      row=r + 1, col=c + 1)
    a = aucs[m]
    fig.layout.annotations[i].text = (f"<b>{i + 1}. {ROTULOS[m]}</b><br><span style='font-size:11px;color:{CINZA}'>"
                                      f"{'maior' if a > 0.5 else 'menor'} em quem cancelou · AUC {br(max(a, 1 - a))}</span>")
fig.update_xaxes(showticklabels=False)
fig.update_layout(height=970, margin=dict(t=150), legend=dict(y=1.05),
                  title="Média dos 3 últimos meses por cliente — métricas ordenadas pela separação entre os grupos<br>"
                        f"<span style='font-size:12px;color:{CINZA}'>Cancelou: 3 meses antes da saída · Ativo: abr–jun/2026 · "
                        "AUC 1,00 = separação perfeita, 0,50 = nenhuma</span>")
fig.show()

## 5. NPS

In [8]:
tri = (nps.groupby(["status", "trimestre_relativo"], observed=True)
          .agg(taxa=("respondeu", lambda s: 100 * s.mean()), nota=("nota_nps", "mean"), n=("respondeu", "size"),
               n_resp=("respondeu", "sum"))
          .reset_index())
TRI = sorted(tri.trimestre_relativo.unique())
rot_tri = {t: f"{3 * t} a {3 * t + 2}".replace("-", "−") for t in TRI}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["Taxa de resposta à pesquisa (%)", "Nota média de quem respondeu (0–10)"])
for c, (col, fmt, extra) in enumerate([("taxa", ".0f", "%"), ("nota", ".1f", "")], start=1):
    for st in ORDEM_STATUS:
        q = tri[tri.status == st]
        fig.add_trace(go.Scatter(x=q.trimestre_relativo, y=q[col], mode="lines+markers", name=st, legendgroup=st,
                                 showlegend=c == 1, line=dict(color=CORES[st], width=2), marker_size=8,
                                 customdata=np.c_[q.trimestre_relativo.map(rot_tri), q.n_resp, q.n],
                                 hovertemplate=f"<b>{st}</b> · meses %{{customdata[0]}}<br>%{{y:{fmt}}}{extra}"
                                               "<br>%{customdata[1]} de %{customdata[2]} responderam<extra></extra>"),
                      row=1, col=c)
fig.update_xaxes(tickvals=TRI, ticktext=[rot_tri[t] for t in TRI], title_text="Meses até a saída (janelas de 3 meses)")
fig.update_yaxes(range=[0, 105], ticksuffix="%", row=1, col=1)
fig.update_yaxes(range=[0, 10], row=1, col=2)
fig.update_layout(height=430, title="NPS trimestral alinhado pela saída: resposta e nota de quem cancelou × ativos")
fig.show()

In [9]:
ultima = nps.sort_values("mes_ref").groupby("cliente_id").tail(1)
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["Todas as pesquisas", "Última pesquisa antes da saída"])
for c, base in enumerate([nps, ultima], start=1):
    dist = pd.crosstab(base.status, base.classificacao_nps, normalize="index").reindex(index=ORDEM_STATUS[::-1], columns=ORDEM_NPS) * 100
    cont = pd.crosstab(base.status, base.classificacao_nps).reindex(index=ORDEM_STATUS[::-1], columns=ORDEM_NPS)
    for k in ORDEM_NPS:
        fig.add_trace(go.Bar(y=dist.index, x=dist[k], orientation="h", name=k, legendgroup=k, showlegend=c == 1,
                             marker=dict(color=CORES_NPS[k], line=dict(color="white", width=2),
                                         pattern_shape="/" if k == "Sem resposta" else "", pattern_fgcolor="#8a8984"),
                             text=[f"{v:.0f}%" if v >= 6 else "" for v in dist[k]], textposition="inside",
                             insidetextanchor="middle", customdata=cont[k],
                             hovertemplate=f"<b>%{{y}}</b> · {k}<br>%{{x:.1f}}% (%{{customdata}} pesquisas)<extra></extra>"),
                      row=1, col=c)
fig.update_layout(barmode="stack", height=320, legend=dict(traceorder="normal"),
                  title="Classificação NPS por status")
fig.update_xaxes(range=[0, 100], ticksuffix="%")
fig.show()

## 6. Mapa de calor da carteira

In [10]:
uso_rec = at[at.mes_relativo >= -3].groupby("cliente_id").uso_plataforma_pct.mean().rename("uso_rec")
ordem = cli.join(uso_rec, on="cliente_id").sort_values(["status", "mes_saida", "uso_rec"]).cliente_id.tolist()
z = at.assign(i=at.mes_ref.map(idx)).pivot(index="cliente_id", columns="i", values="uso_plataforma_pct").reindex(ordem)
rotulos = np.tile([fmt_mes(p) for p in MESES], (len(ordem), 1))
LO, MID, HI = 40, 80, 100
fig = go.Figure(go.Heatmap(z=z.values, x=list(z.columns), y=ordem, zmin=LO, zmax=HI, xgap=1, ygap=1, customdata=rotulos,
                           colorscale=[[0, "#b42f2f"], [(MID - LO) / (HI - LO), "#efeeea"], [1, "#1c5cab"]],
                           colorbar=dict(title="Uso (%)", ticksuffix="%", len=0.4, y=1, yanchor="top", thickness=14),
                           hovertemplate="<b>%{y}</b> · %{customdata}<br>Uso: %{z:.1f}%<extra></extra>"))
cx = cli.set_index("cliente_id").loc[[c for c in ordem if c in set(canc.cliente_id)]]
fig.add_trace(go.Scatter(x=cx.mes_cancelamento.map(idx), y=cx.index, mode="markers", name="Mês do cancelamento",
                         marker=dict(symbol="x-thin", size=9, line=dict(width=2, color="#2b2b29")),
                         customdata=cx.mes_cancelamento.map(fmt_mes),
                         hovertemplate="<b>%{y}</b> cancelou em %{customdata}<extra></extra>"))
n_c = len(canc)
fig.add_hline(y=n_c - 0.5, line=dict(color="#2b2b29", width=1.5))
for y, txt in [((n_c - 1) / 2, f"Cancelaram ({n_c})"), ((n_c + len(ordem) - 1) / 2, f"Ativos ({len(ordem) - n_c})<br>menor uso recente no topo")]:
    fig.add_annotation(x=1.005, xref="paper", y=y, text=txt, showarrow=False, textangle=-90, xanchor="left", font_size=12)
eixo_meses(fig, tickangle=-45, side="top")
fig.update_yaxes(autorange="reversed", tickfont_size=9, showgrid=False)
fig.update_layout(height=1350, margin=dict(r=110, t=150), legend=dict(y=1.0, x=1.09, xanchor="left", orientation="v"),
                  title="Uso da plataforma por cliente e mês — quem cancelou esfria antes de sair (vazio = após a saída)")
fig.show()

## 7. Correlação entre métricas

In [11]:
corr = at[METRICAS].corr("spearman")
corr = corr.where(np.tril(np.ones(corr.shape, dtype=bool), k=-1)).iloc[1:, :-1]
nomes_y, nomes_x = [ROTULOS[m] for m in corr.index], [ROTULOS[m] for m in corr.columns]
fig = go.Figure(go.Heatmap(z=corr.values, x=nomes_x, y=nomes_y, zmin=-1, zmax=1, xgap=2, ygap=2,
                           colorscale=[[0, "#b42f2f"], [0.5, "#efeeea"], [1, "#1c5cab"]],
                           text=corr.map(lambda v: "" if pd.isna(v) else br(v)).values, texttemplate="%{text}",
                           colorbar=dict(title="ρ", thickness=14),
                           hovertemplate="%{y} × %{x}<br>ρ = %{z:.2f}<extra></extra>"))
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(tickangle=-35, showgrid=False)
fig.update_layout(height=620, width=900, title="Correlação de Spearman entre as métricas mensais (todas as linhas cliente × mês)")
fig.show()

## 8. Explorador por cliente

In [12]:
PAINEL = ["uso_plataforma_pct", "chamados_abertos", "tempo_medio_resolucao_h",
          "pct_sla_cumprido", "dias_atraso_pagamento", "nota_nps"]
ref = at[at.status == "Ativo"].groupby("mes_ref")[PAINEL[:-1]].median()
ref["nota_nps"] = nps[nps.status == "Ativo"].groupby("mes_ref").nota_nps.median()
cli_i = cli.set_index("cliente_id")

def painel_cliente(cid):
    c, a, n = cli_i.loc[cid], at[at.cliente_id == cid], nps[nps.cliente_id == cid]
    cor = CORES[c.status]
    fig = make_subplots(rows=2, cols=3, shared_xaxes=True, vertical_spacing=0.14, horizontal_spacing=0.07,
                        subplot_titles=[f"<b>{ROTULOS[m]}</b>" for m in PAINEL])
    for i, m in enumerate(PAINEL):
        r, col = i // 3 + 1, i % 3 + 1
        rr = ref[m].dropna()
        fig.add_trace(go.Scatter(x=rr.index.map(idx), y=rr, mode="lines", name="Mediana dos ativos", legendgroup="ref",
                                 showlegend=i == 0, line=dict(color=CINZA, width=1.5, dash="dot"), customdata=rr.index.map(fmt_mes),
                                 hovertemplate="Mediana dos ativos · %{customdata}: %{y:,.1f}<extra></extra>"), row=r, col=col)
        s = (n[n.respondeu] if m == "nota_nps" else a)[["mes_ref", m]].dropna()
        fig.add_trace(go.Scatter(x=s.mes_ref.map(idx), y=s[m], mode="lines+markers", name=cid, legendgroup="cli",
                                 showlegend=i == 0, line=dict(color=cor, width=2.5), marker_size=7,
                                 customdata=s.mes_ref.map(fmt_mes),
                                 hovertemplate=f"<b>{cid}</b> · %{{customdata}}<br>{ROTULOS[m]}: %{{y:,.1f}}<extra></extra>"),
                      row=r, col=col)
        if c.status == "Cancelou":
            fig.add_vline(x=idx(c.mes_cancelamento), line=dict(color=CORES["Cancelou"], dash="dash", width=1.5), row=r, col=col)
    sem = n[~n.respondeu]
    fig.add_trace(go.Scatter(x=sem.mes_ref.map(idx), y=[0] * len(sem), mode="markers", name="NPS sem resposta",
                             marker=dict(symbol="x-thin", size=12, line=dict(width=2.5, color="#52514e")),
                             customdata=sem.mes_ref.map(fmt_mes), hovertemplate="%{customdata}: sem resposta<extra></extra>"),
                  row=2, col=3)
    fig.update_yaxes(range=[-0.8, 10.5], row=2, col=3)
    eixo_meses(fig, passo=3)
    situacao = f"Cancelou em {fmt_mes(c.mes_cancelamento)}" if c.status == "Cancelou" else "Ativo"
    fig.update_layout(height=620, hovermode="closest",
                      title=f"<b>{cid}</b> · {c.segmento} · {c.porte} · {c.plano} · {fmt_brl(c.valor_mensal)}/mês · <b>{situacao}</b>")
    return fig

opcoes = [(f"{r.cliente_id} · " + (f"Cancelou em {fmt_mes(r.mes_cancelamento)}" if r.status == "Cancelou" else "Ativo")
           + f" · {fmt_brl(r.valor_mensal)}/mês", r.cliente_id)
          for r in cli.sort_values(["status", "mes_cancelamento", "cliente_id"]).itertuples()]
seletor = widgets.Dropdown(options=opcoes, description="Cliente", layout=widgets.Layout(width="460px"))
saida = widgets.Output()

def atualizar(_=None):
    with saida:
        saida.clear_output(wait=True)
        painel_cliente(seletor.value).show()

seletor.observe(atualizar, names="value")
atualizar()
display(seletor, saida)

Dropdown(description='Cliente', layout=Layout(width='460px'), options=(('C022 · Cancelou em mai/2025 · R$ 13.1…

Output()